# 🚀 Kaggle Dual-GPU LLM Automated Unit Test Generation
### Zero-Cost ($0 API Cost) Benchmarking using 2x Tesla T4 (Dual-GPU MoA)

This notebook demonstrates automated test generation and quality gate evaluation (Branch Coverage + Mutation Score) using local open-weight models on **Kaggle Dual-GPU (2x Tesla T4 15GB)**:
- **GPU 0 (cuda:0)**: `Qwen/Qwen2.5-Coder-14B-Instruct` (4-bit NF4, ~8.5GB VRAM) — Coder model.
- **GPU 1 (cuda:1)**: `Qwen/Qwen3-4B-Instruct-2507` (4-bit NF4, ~3.2GB VRAM) — Critic / Verifier model.

---
### ⚙️ Kaggle Settings Check:
1. **Accelerator**: Select `GPU T4 x2` in the right-hand panel.
2. **Internet**: Toggle `Internet On` (required for downloading Hugging Face models and packages).
3. **Persistence**: `Files only`.

In [ ]:
# Cell 1: Install Required Libraries
!pip install -q --upgrade pip
!pip install -q torch transformers accelerate bitsandbytes coverage pytest
print("✅ Libraries installed successfully!")

In [ ]:
# Cell 2: Hardware & Dual-GPU Verification
import torch

print("=" * 60)
print("🔍 Hardware Diagnostics:")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"GPU Count: {gpu_count}")

if gpu_count < 2:
    print("⚠️ Notice: Less than 2 GPUs detected. Set Accelerator to 'GPU T4 x2' in Kaggle settings.")
    print("If running on 1 GPU, both models will run on cuda:0 sequentially.")
else:
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  - GPU {i}: {props.name} | Total Memory: {props.total_memory / (1024**3):.2f} GB")
print("=" * 60)

In [ ]:
# Cell 3: Clone Repository or Setup Target Project
import os
import subprocess
from pathlib import Path

WORKDIR = Path("/kaggle/working/LLM-analysis")
if not WORKDIR.exists():
    print("📥 Cloning LLM-analysis repository...")
    !git clone https://github.com/nguyendevwk/LLM-analysis.git /kaggle/working/LLM-analysis

os.chdir(WORKDIR)
print(f"📍 Current working directory: {os.getcwd()}")
assert (WORKDIR / "order_calculator.py").exists(), "order_calculator.py not found!"
print("✅ Project repository loaded successfully!")

In [ ]:
# Cell 4: Initialize Dual-GPU MoA Engine
import sys
sys.path.insert(0, str(WORKDIR))
sys.path.insert(0, str(WORKDIR / "automation"))
sys.path.insert(0, str(WORKDIR / "kaggle"))

from kaggle.kaggle_dual_gpu_runner import DualGPUEngine, run_council_pipeline

CODER_MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
CRITIC_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Initializing Engine:\n - Coder: {CODER_MODEL} (GPU 0)\n - Critic: {CRITIC_MODEL} (GPU 1)")
engine = DualGPUEngine(
    coder_model_id=CODER_MODEL,
    critic_model_id=CRITIC_MODEL,
    load_in_4bit=True
)
print("✅ DualGPUEngine initialized and ready for inference!")

In [ ]:
# Cell 5: Execute Council Test Generation & Quality Gate Evaluation
source_file = WORKDIR / "order_calculator.py"
output_dir = Path("/kaggle/working/evaluation_results")

print("🚀 Starting Council Automated Test Generation...")
summary = run_council_pipeline(
    engine=engine,
    source_file=source_file,
    class_name="OrderCalculator",
    context_level="full_context",
    output_dir=output_dir
)

gates = summary["quality_gates"]
telemetry = summary["telemetry"]

print("\n" + "=" * 65)
print("📊 BENCHMARK & QUALITY GATE RESULTS (KAGGLE DUAL-T4)")
print("=" * 65)
print(f"Statement Coverage: {gates['statement_coverage']:.1f}%")
print(f"Branch Coverage:    {gates['branch_coverage']:.1f}%")
print(f"Mutation Score:     {gates['mutation_score']:.1f}% ({gates['mutants_killed']}/{gates['mutants_total']} mutants killed)")
print(f"Quality Score:      {gates['quality_score']:.1f} / 100")
print(f"Tests Passed:       {gates['tests_passed']} / {gates['total_tests']}")
print(f"Total Tokens:       {telemetry['total_tokens']}")
print(f"Generation Time:    {telemetry['total_generation_seconds']:.2f} s")
print("=" * 65)

In [ ]:
# Cell 6: Export Results & Archive for Download
import shutil

zip_path = "/kaggle/working/kaggle_benchmark_results.zip"
shutil.make_archive("/kaggle/working/kaggle_benchmark_results", "zip", str(output_dir))
print(f"📦 Downloadable archive created: {zip_path}")

# Preview generated test suite snippet
final_tests = (output_dir / "final_tests.py").read_text()
print("\n📄 Preview Generated Test Code (First 25 lines):")
print("\n".join(final_tests.splitlines()[:25]))